# Modelo GRU — Red recurrente con compuertas (apilada)
## TFG: Predicción de Tráfico Urbano (M30 + URB)

**Input:** `data/train.parquet`, `data/val.parquet`, `data/test.parquet`
**Targets:** `intensidad_trafico`, `ocupacion`, `carga`

---

### Por qué GRU y cómo se compara

Comparte **exactamente el mismo pipeline de datos** que `MMMModel.ipynb` (Fases 1-3)
y que `Modelo2.0.ipynb`: las mismas 22 features, el mismo split temporal 70/15/15
sin shuffle, la misma ventana de **12 pasos (3 h)** y la misma normalización. Lo
único que cambia es el bloque recurrente, de modo que la comparación es justa.

La GRU es una recurrente con compuertas como la LSTM pero con **una compuerta menos
y sin estado de celda separado**: menos parámetros, entrenamiento más rápido y, a
menudo, rendimiento equivalente en series con dependencias de medio plazo. Aquí se
usa **apilada y unidireccional** `GRU(128) → GRU(64)`, para contrastar con el
BiLSTM bidireccional del Modelo 2.0.

| Aspecto | BiLSTM v2 | GRU (este notebook) |
|---|---|---|
| Bloque recurrente | Bidirectional LSTM(128)→LSTM(64) | **GRU(128) → GRU(64)** (unidireccional) |
| Dirección | bidireccional | unidireccional (causal, como inferencia real) |
| Parámetros | mayor | **menor** |
| Resto (datos, loss Huber, AMP, ckpt) | — | idéntico |

Salidas (gráficas + métricas JSON) en `SALIDAS modelo GRU/`.

In [2]:
## 0. Imports y configuración global
import os
import gc
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings("ignore")

# Reproducibilidad
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# --- GPU AMD vía ROCm ---
# En PyTorch-ROCm la GPU AMD se expone bajo el namespace 'cuda' (HIP),
# así que torch.cuda.is_available() y .to("cuda") apuntan a la Radeon.
DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"        # mixed precision fp16 (equivalente a mixed_float16 de v1)
if USE_AMP:
    torch.backends.cudnn.benchmark = True

# Directorios de salida
OUT_DIR_PLOTS  = "SALIDAS modelo GRU"
OUT_DIR_MODELS = OUT_DIR_PLOTS          # los checkpoints .pt se guardan junto a las gráficas/métricas
os.makedirs(OUT_DIR_PLOTS,  exist_ok=True)
os.makedirs(OUT_DIR_MODELS, exist_ok=True)

# Constantes
TARGET_COLS = ["intensidad_trafico", "ocupacion", "carga"]
SEQ_LEN     = 12                                # 12 × 15 min = 3 h de historial
SEQ_STEPS   = list(range(SEQ_LEN, 0, -1))       # [12, 11, ..., 1]: antiguo → reciente

print(f"PyTorch {torch.__version__}")
if DEVICE.type == "cuda":
    print(f"GPU      : {torch.cuda.get_device_name(0)}  (HIP {torch.version.hip})")
else:
    print("GPU      : no disponible — ejecutando en CPU")
print(f"Dispositivo: {DEVICE}   | mixed precision fp16: {USE_AMP}")
print(f"SEQ_LEN = {SEQ_LEN} pasos ({SEQ_LEN*15} min = {SEQ_LEN/4:.1f} h de historial)")

PyTorch 2.5.1+rocm6.2
GPU      : AMD Radeon RX 6700 XT  (HIP 6.2.41133-dd7f95766)
Dispositivo: cuda   | mixed precision fp16: True
SEQ_LEN = 12 pasos (180 min = 3.0 h de historial)


## 1. Carga de los splits temporales

Cargo los parquets generados en la Fase 3 del notebook original. Ya contienen las 22 features de contexto y los `lag1..lag4` de los 3 targets. El split es temporal 70/15/15 sin shuffle ni solapamiento.

In [ ]:
# Rutas de los splits y de la caché de arrays compactos
TRAIN_PATH = "data/train.parquet"
VAL_PATH   = "data/val.parquet"
TEST_PATH  = "data/test.parquet"
CACHE_DIR  = os.path.expanduser("~/.cache/prediccion-trafico/seq_cache")  # en NVMe (rápido), no en el USB
os.makedirs(CACHE_DIR, exist_ok=True)
print("Splits :", TRAIN_PATH, VAL_PATH, TEST_PATH)
print("Caché  :", CACHE_DIR)

## 2. Ampliación de lags (4 → 12) con caché en disco — frugal en RAM

Los parquets traen `lag1..lag4`; para `SEQ_LEN=12` hacen falta los 12 lags. Cargar
los tres splits a la vez (37,5 M filas × 42 columnas) y duplicar el frame al ordenar
**desbordaba los 15 GB de RAM** (el kernel moría por OOM).

Ahora se procesa **un split cada vez**, leyendo **solo las 24 columnas necesarias**
(de las 42), y se **siembra** cada split con la cola (últimas 12 filas/sensor) del
anterior para no perder continuidad. Los arrays compactos resultantes se **guardan
en `~/.cache/prediccion-trafico/seq_cache/*.npy (en el NVMe, no en el USB)`**: se calculan una sola vez y los tres notebooks (BiLSTM v2,
GRU, Transformer) los reutilizan. `test` se carga con `mmap` porque solo se lee en la
evaluación final. Pico de RAM ≈ 6-7 GB en vez de >17 GB.

In [ ]:
# 19 features de contexto (constantes a lo largo de la secuencia)
import pyarrow.parquet as pq
from numpy.lib.format import open_memmap

CONTEXT_COLS = [
    "slot_sin", "slot_cos",
    "dia_sem_sin", "dia_sem_cos",
    "mes_sin", "mes_cos",
    "es_laborable",
    "lluvia_ord", "precip_mm",
    "hay_accidente", "accidente_reciente_1h", "tiempo_accidente_norm",
    "tipo_elem",
    "intensidad_trafico_roll4", "intensidad_trafico_roll8",
    "ocupacion_roll4",          "ocupacion_roll8",
    "carga_roll4",              "carga_roll8",
]
N_CTX      = len(CONTEXT_COLS)
N_FEATURES = 3 + N_CTX                       # 22 (idéntico a v1)
NEED_COLS  = ["id", "datetime"] + TARGET_COLS + CONTEXT_COLS   # solo 24 de las 42
N_CHUNKS   = 8                               # bloques de sensores (limita el pico de RAM)

def build_split_chunked(path, seed_tail):
    """Construye los arrays compactos de un split SIN cargarlo entero en RAM.
    Trocea por sensores: lee del parquet solo los sensores de cada bloque
    (filtro pushdown) y solo las 24 columnas útiles, calcula los 12 lags y
    ESCRIBE directamente a memmaps en disco (lag/ctx/y), así el pico de RAM es
    el de un bloque (~1-2 GB), no el del split (~9 GB que provocaba el OOM).
    Devuelve (count_real, meta[id,datetime], tail_para_el_siguiente_split)."""
    ids = np.sort(pd.read_parquet(path, columns=["id"])["id"].unique())
    chunks = np.array_split(ids, N_CHUNKS)
    n_rows = pq.ParquetFile(path).metadata.num_rows
    # memmaps dimensionados al máximo (n_rows); luego se recorta con el count real
    split = os.path.splitext(os.path.basename(path))[0]
    lag_mm = open_memmap(f"{CACHE_DIR}/lag_{split}.npy", mode="w+", dtype="float32", shape=(n_rows, SEQ_LEN, 3))
    ctx_mm = open_memmap(f"{CACHE_DIR}/ctx_{split}.npy", mode="w+", dtype="float32", shape=(n_rows, N_CTX))
    y_mm   = open_memmap(f"{CACHE_DIR}/y_{split}.npy",   mode="w+", dtype="float32", shape=(n_rows, 3))
    metas, next_tails, off = [], [], 0
    for ch in chunks:
        ch_list = [int(x) for x in ch]
        df = pd.read_parquet(path, columns=NEED_COLS, engine="pyarrow",
                             filters=[("id", "in", ch_list)])
        df = df.sort_values(["id", "datetime"]).reset_index(drop=True)
        ctx  = df[CONTEXT_COLS].to_numpy("float32")
        y    = df[TARGET_COLS].to_numpy("float32")
        meta = df[["id", "datetime"]]
        next_tails.append(df.groupby("id", observed=True).tail(SEQ_LEN)[["id", "datetime"] + TARGET_COLS])
        base = df[["id", "datetime"] + TARGET_COLS].copy(); base["_real"] = True
        del df
        if seed_tail is not None:
            seed = seed_tail[seed_tail["id"].isin(ch_list)].copy(); seed["_real"] = False
            base = pd.concat([seed, base], ignore_index=True)
        base = base.sort_values(["id", "datetime"]).reset_index(drop=True)
        realmask = base["_real"].to_numpy()
        g = base.groupby("id", observed=True)
        lag = np.empty((int(realmask.sum()), SEQ_LEN, 3), dtype="float32")
        for j, t in enumerate(TARGET_COLS):
            for step_idx, k in enumerate(SEQ_STEPS):
                lag[:, step_idx, j] = g[t].shift(k).to_numpy("float32")[realmask]
        del base, g
        valid = ~np.isnan(lag).any(axis=(1, 2))        # descarta primeras 12/sensor sin histórico
        lag = lag[valid]; ctx = ctx[valid]; y = y[valid]; meta = meta.loc[valid]
        m = len(lag)
        lag_mm[off:off + m] = lag; ctx_mm[off:off + m] = ctx; y_mm[off:off + m] = y
        metas.append(meta); off += m
        del lag, ctx, y; gc.collect()
    lag_mm.flush(); ctx_mm.flush(); y_mm.flush()
    del lag_mm, ctx_mm, y_mm; gc.collect()
    meta_all  = pd.concat(metas, ignore_index=True)
    next_tail = pd.concat(next_tails, ignore_index=True)
    return off, meta_all, next_tail

CNT_PATH = f"{CACHE_DIR}/counts.json"
CACHE_OK = (all(os.path.exists(f"{CACHE_DIR}/{a}_{s}.npy")
                for s in ("train", "val", "test") for a in ("lag", "ctx", "y"))
            and os.path.exists(f"{CACHE_DIR}/meta_test.parquet")
            and os.path.exists(CNT_PATH))

if not CACHE_OK:
    print("Caché no encontrada -> construyendo arrays compactos (una sola vez, por bloques de sensores)...")
    counts = {}
    counts["train"], _,         tail = build_split_chunked(TRAIN_PATH, None); print(f"  train OK  ({counts['train']:,} filas)")
    counts["val"],   _,         tail = build_split_chunked(VAL_PATH,   tail); print(f"  val   OK  ({counts['val']:,} filas)")
    counts["test"],  meta_test, _    = build_split_chunked(TEST_PATH,  tail); print(f"  test  OK  ({counts['test']:,} filas)")
    meta_test.to_parquet(f"{CACHE_DIR}/meta_test.parquet", index=False)
    with open(CNT_PATH, "w") as f: json.dump(counts, f)
    del meta_test, tail; gc.collect()
    print(f"  caché escrita en {CACHE_DIR}")
else:
    with open(CNT_PATH) as f: counts = json.load(f)
    print("Caché encontrada -> se omite el cálculo pandas (carga directa).")

# --- Carga con mmap (RSS mínima): el SO cachea desde el NVMe; el shuffle del
#     DataLoader hace lecturas aleatorias rápidas. Sin copia a RAM -> sin riesgo de OOM. ---
def _load(name, split):
    return np.load(f"{CACHE_DIR}/{name}_{split}.npy", mmap_mode="r")[:counts[split]]

lag_train = _load("lag", "train"); ctx_train = _load("ctx", "train"); y_train = np.array(_load("y", "train"))
lag_val   = _load("lag", "val");   ctx_val   = _load("ctx", "val");   y_val   = np.array(_load("y", "val"))
lag_test  = _load("lag", "test");  ctx_test  = _load("ctx", "test");  y_test  = np.array(_load("y", "test"))
test_df   = pd.read_parquet(f"{CACHE_DIR}/meta_test.parquet")   # id+datetime para las gráficas

print(f"train {lag_train.shape}  val {lag_val.shape}  test {lag_test.shape}")
gc.collect()

## 3. Secuencias — almacenamiento compacto y ensamblado en GPU

El tensor denso `(n, 12, 22)` ocuparía ~27 GB. Como **19 de las 22 features son contexto constante** a lo largo de la secuencia y solo las 3 de lag varían por paso, en la celda anterior se guardó por split únicamente:

- `lag_*`: `(n, 12, 3)` — los 3 targets en los 12 pasos (orden `SEQ_STEPS`).
- `ctx_*`: `(n, 19)` — las 19 features de contexto, una vez por fila.

Total ~5,5 GB en vez de ~27 GB. La secuencia completa `(B, 12, 22)` se **ensambla y normaliza por lote dentro de la GPU** (función `assemble`, celda de entrenamiento), nunca en RAM.

In [ ]:
# Resumen de los arrays compactos (ya construidos por split en la celda anterior)
mem_compact = sum(a.nbytes for a in (lag_train, ctx_train, lag_val, ctx_val, lag_test, ctx_test)) / 1e9
mem_denso   = (len(lag_train) + len(lag_val) + len(lag_test)) * SEQ_LEN * N_FEATURES * 4 / 1e9
print(f"lag_train {lag_train.shape} + ctx_train {ctx_train.shape}  → secuencia efectiva (n, {SEQ_LEN}, {N_FEATURES})")
print(f"filas train/val/test : {len(lag_train):,} / {len(lag_val):,} / {len(lag_test):,}")
print(f"RAM arrays compactos : {mem_compact:.1f} GB   (el tensor denso habría sido ~{mem_denso:.0f} GB)")

## 4. Normalización (equivalente a `StandardScaler`, sin materializar el tensor)

Calculo media y desviación típica de las 22 features **exactamente como haría `StandardScaler` sobre el tensor denso**, pero a partir de los arrays compactos:

- Las **3 features de lag** se normalizan agrupando sus 12 pasos (la misma feature aparece 12 veces con valores `lag1..lag12`): media/σ sobre `lag_train` en los ejes `(filas, pasos)`.
- Las **19 de contexto** se normalizan por columna (repetirlas 12× no cambia su media/σ).

`σ=0` se sustituye por 1 (igual que `StandardScaler`). Los estadísticos se suben a la GPU (`X_MEAN`, `X_STD`) y se aplican por lote. `y` mantiene su `StandardScaler` (es pequeño); `y_test` se deja en escala real.

In [ ]:
# Estadísticos de normalización SIN construir el tensor denso (mismos que StandardScaler)
mean_ = np.empty(N_FEATURES, dtype="float32")
std_  = np.empty(N_FEATURES, dtype="float32")
mean_[:3] = lag_train.mean(axis=(0, 1))      # 3 lags: agrupando los 12 pasos
std_[:3]  = lag_train.std(axis=(0, 1))       # std poblacional (ddof=0), como StandardScaler
mean_[3:] = ctx_train.mean(axis=0)           # 19 contexto: por columna
std_[3:]  = ctx_train.std(axis=0)
std_ = np.where(std_ == 0, 1.0, std_).astype("float32")

scaler_y = StandardScaler()
y_train_n = scaler_y.fit_transform(y_train).astype("float32")
y_val_n   = scaler_y.transform(y_val).astype("float32")
# y_test se mantiene SIN normalizar (métricas en escala real)

# Estadísticos en GPU para normalizar cada lote tras ensamblarlo
X_MEAN = torch.tensor(mean_, device=DEVICE)
X_STD  = torch.tensor(std_,  device=DEVICE)

print(f"X mean[:3]={np.round(mean_[:3],3)}  std[:3]={np.round(std_[:3],3)}")
print(f"y train   : media={y_train_n.mean():.4f}  std={y_train_n.std():.4f}")
gc.collect()

## 5. Arquitectura — GRU apilada

```
Input(12, 22)
  → GRU(128, return_sequences=True)
  → Dropout(0.3)
  → GRU(64, return_sequences=False)   # se usa el último estado oculto h_n
  → Dropout(0.3)
  → Dense(64, ReLU)
  → Dense(32, ReLU)
  → Dense(3, lineal)
```

**Loss**: `HuberLoss(delta=1.0)` (igual que el Modelo 2.0). Entrenamiento con
`torch.autocast(fp16)` + `GradScaler` sobre la GPU AMD (ROCm). Unidireccional a
propósito: en producción solo se dispone del pasado, así que esta GRU refleja el
escenario de inferencia real, a diferencia del BiLSTM.

In [ ]:
class GRUNet(nn.Module):
    """GRU apilada unidireccional. La 2ª capa devuelve su estado oculto final
    h_n (resumen de toda la secuencia tras el último paso), análogo a
    return_sequences=False en Keras."""
    def __init__(self, n_features, n_targets=3, dropout=0.3):
        super().__init__()
        self.gru1   = nn.GRU(n_features, 128, batch_first=True)
        self.drop1  = nn.Dropout(dropout)
        self.gru2   = nn.GRU(128, 64, batch_first=True)
        self.drop2  = nn.Dropout(dropout)
        self.dense1 = nn.Linear(64, 64)
        self.dense2 = nn.Linear(64, 32)
        self.out    = nn.Linear(32, n_targets)
        self.act    = nn.ReLU()

    def forward(self, x):
        x, _      = self.gru1(x)            # (n, seq, 128)
        x         = self.drop1(x)
        _, h_n    = self.gru2(x)            # h_n: (1, n, 64)
        x         = h_n[-1]                 # (n, 64) estado final última capa
        x         = self.drop2(x)
        x         = self.act(self.dense1(x))
        x         = self.act(self.dense2(x))
        return self.out(x)

model = GRUNet(N_FEATURES, n_targets=len(TARGET_COLS), dropout=0.4).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nParámetros entrenables: {n_params:,}")

## 6. Entrenamiento (checkpointing reanudable)

Mismos hiperparámetros y bucle que el Modelo 2.0 para que la comparación aísle el
efecto de la arquitectura: batch 1024, Huber(δ=1.0), Adam 1e-4,
ReduceLROnPlateau (patience=7) y EarlyStopping (patience=15). Si el notebook se
interrumpe, al re-ejecutar la celda se reanuda desde `outputs/gru_last.pt`.

In [ ]:
BATCH_SIZE = 1024
EPOCHS     = 100

def assemble(lag_b, ctx_b):
    """Ensambla y normaliza el lote (B, SEQ_LEN, N_FEATURES) en GPU:
    pone los 3 lags por paso y replica el contexto en los 12 pasos."""
    B = lag_b.size(0)
    seq = torch.empty((B, SEQ_LEN, N_FEATURES), device=lag_b.device)
    seq[:, :, :3] = lag_b
    seq[:, :, 3:] = ctx_b.unsqueeze(1).expand(-1, SEQ_LEN, -1)
    return (seq - X_MEAN) / X_STD

# --- DataLoaders sobre los arrays compactos (la secuencia se arma por lote) ---
gen = torch.Generator().manual_seed(SEED)
train_ds = TensorDataset(torch.from_numpy(lag_train), torch.from_numpy(ctx_train), torch.from_numpy(y_train_n))
val_ds   = TensorDataset(torch.from_numpy(lag_val),   torch.from_numpy(ctx_val),   torch.from_numpy(y_val_n))
# num_workers>0 + persistent_workers: la carga/indexado de lotes (CPU) solapa
# con el cómputo en GPU -> sube el uso de GPU (antes ~30%, esperando datos).
NUM_WORKERS = 4
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          drop_last=True, pin_memory=USE_AMP, generator=gen,
                          num_workers=NUM_WORKERS, persistent_workers=True,
                          prefetch_factor=4)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
                          pin_memory=USE_AMP, num_workers=NUM_WORKERS,
                          persistent_workers=True, prefetch_factor=4)

# --- Pérdida, optimizador, scheduler, AMP ---
loss_fn   = nn.HuberLoss(delta=1.0)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=1e-4)  # +regularización vs overfitting
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=4, min_lr=1e-7)
scaler    = torch.amp.GradScaler("cuda", enabled=USE_AMP)
PATIENCE_ES = 15
N_TGT = len(TARGET_COLS)

# --- Rutas de checkpoint (.pt) ---
CKPT_BEST  = os.path.join(OUT_DIR_MODELS, "gru_best.pt")
CKPT_LAST  = os.path.join(OUT_DIR_MODELS, "gru_last.pt")
STATE_PATH = os.path.join(OUT_DIR_MODELS, "gru_state.json")

def run_epoch(loader, train_mode):
    """Una época. Las métricas se acumulan COMO TENSORES EN LA GPU y solo se
    bajan a CPU una vez al final de la época. Así se evita el sincronizado
    GPU->CPU por lote (.item()/.cpu()), que con ~25k lotes/epoch estrangulaba
    la GPU (uso bajo en rocm-smi). Mismas métricas: loss(Huber), MAE, RMSE y R²
    (R² = media uniforme por target, como tf.keras.metrics.R2Score)."""
    model.train(train_mode)
    tot_loss = torch.zeros((),    device=DEVICE)
    sum_abs  = torch.zeros(N_TGT, device=DEVICE)
    sum_res2 = torch.zeros(N_TGT, device=DEVICE)
    sum_y    = torch.zeros(N_TGT, device=DEVICE)
    sum_y2   = torch.zeros(N_TGT, device=DEVICE)
    tot_n = 0
    torch.set_grad_enabled(train_mode)
    for lag_b, ctx_b, yb in loader:
        lag_b = lag_b.to(DEVICE, non_blocking=True)
        ctx_b = ctx_b.to(DEVICE, non_blocking=True)
        yb    = yb.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            out  = model(assemble(lag_b, ctx_b))
            loss = loss_fn(out, yb)
        if train_mode:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        bs = yb.size(0); tot_n += bs
        with torch.no_grad():
            p = out.float(); t = yb.float()
            tot_loss += loss.detach() * bs
            sum_abs  += (p - t).abs().sum(0)
            sum_res2 += ((p - t) ** 2).sum(0)
            sum_y    += t.sum(0); sum_y2 += (t ** 2).sum(0)
    torch.set_grad_enabled(True)
    # único traslado GPU->CPU de toda la época
    sum_abs_  = sum_abs.cpu().numpy()
    sum_res2_ = sum_res2.cpu().numpy()
    sum_y_    = sum_y.cpu().numpy()
    sum_y2_   = sum_y2.cpu().numpy()
    mae    = float(sum_abs_.sum()  / (tot_n * N_TGT))
    rmse   = float(np.sqrt(sum_res2_.sum() / (tot_n * N_TGT)))
    ss_tot = sum_y2_ - (sum_y_ ** 2) / tot_n
    r2     = float(np.mean(1.0 - sum_res2_ / np.where(ss_tot == 0, 1.0, ss_tot)))
    return {"loss": float(tot_loss.cpu()) / tot_n, "mae": mae, "rmse": rmse, "r2": r2}

# --- Estado + reanudación ---
initial_epoch = 0
hist = {k: [] for k in ("loss", "mae", "rmse", "r2",
                        "val_loss", "val_mae", "val_rmse", "val_r2")}
best_val   = float("inf")
best_state = None
es_counter = 0

if os.path.exists(CKPT_LAST):
    ckpt = torch.load(CKPT_LAST, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    scaler.load_state_dict(ckpt["scaler"])
    initial_epoch = ckpt["epoch"]
    hist          = ckpt["history"]
    best_val      = ckpt["best_val"]
    es_counter    = ckpt["es_counter"]
    best_state    = ckpt.get("best_state")
    if initial_epoch >= EPOCHS:
        print(f"[checkpoint] epoch {initial_epoch} >= EPOCHS={EPOCHS}. "
              f"Borra {CKPT_LAST} para reentrenar.")
    else:
        print(f"[checkpoint] reanudando desde epoch {initial_epoch}")
else:
    print("[checkpoint] inicio desde cero (sin checkpoint previo)")

try:
    for epoch in range(initial_epoch, EPOCHS):
        tr = run_epoch(train_loader, True)
        va = run_epoch(val_loader,  False)
        for k in ("loss", "mae", "rmse", "r2"):
            hist[k].append(tr[k]); hist["val_" + k].append(va[k])

        prev_lr = optimizer.param_groups[0]["lr"]
        scheduler.step(va["loss"])
        new_lr = optimizer.param_groups[0]["lr"]
        lr_msg = f"  | lr {prev_lr:.1e}->{new_lr:.1e}" if new_lr < prev_lr else ""

        improved = va["loss"] < best_val
        if improved:
            best_val = va["loss"]; es_counter = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            torch.save(best_state, CKPT_BEST)
        else:
            es_counter += 1

        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                    "epoch": epoch + 1, "history": hist, "best_val": best_val,
                    "es_counter": es_counter, "best_state": best_state}, CKPT_LAST)
        with open(STATE_PATH, "w") as f:
            json.dump({"last_epoch": epoch + 1, "history": hist, "best_val": best_val}, f)

        flag = "  *mejor*" if improved else ""
        print(f"Epoch {epoch+1:3d}/{EPOCHS}  loss {tr['loss']:.4f}  "
              f"val_loss {va['loss']:.4f}  val_mae {va['mae']:.4f}  "
              f"val_rmse {va['rmse']:.4f}  val_r2 {va['r2']:.4f}{lr_msg}{flag}")

        if es_counter >= PATIENCE_ES:
            print(f"\n[early stopping] sin mejora en {PATIENCE_ES} epochs. "
                  f"Restaurando mejores pesos (val_loss={best_val:.4f}).")
            break
except KeyboardInterrupt:
    print("\n[interrumpido] checkpoint guardado en gru_last.pt. "
          "Re-ejecuta la celda para reanudar.")

if best_state is not None:
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

class _History:
    def __init__(self, h): self.history = h
history = _History(hist)
print(f"\nEntrenamiento finalizado. Epochs totales: {len(history.history['loss'])}  "
      f"| mejor val_loss: {best_val:.4f}")

## 7. Evaluación en test

Predicción en escala normalizada → `inverse_transform` con `scaler_y` para obtener las unidades reales de cada target. Métricas MAE y RMSE en escala real.

In [ ]:
# Predicción en test (escala normalizada) → vuelta a escala original
model.eval()
test_ds = TensorDataset(torch.from_numpy(lag_test), torch.from_numpy(ctx_test))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False, pin_memory=USE_AMP)
preds = []
with torch.no_grad():
    for lag_b, ctx_b in test_loader:
        lag_b = lag_b.to(DEVICE, non_blocking=True)
        ctx_b = ctx_b.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            out = model(assemble(lag_b, ctx_b))
        preds.append(out.float().cpu().numpy())
y_pred_norm = np.concatenate(preds, axis=0)
y_pred   = scaler_y.inverse_transform(y_pred_norm)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

cur_metrics = {}
for i, target in enumerate(TARGET_COLS):
    cur_metrics[target] = {
        "MAE":  float(mean_absolute_error(y_test[:, i], y_pred[:, i])),
        "RMSE": rmse(y_test[:, i], y_pred[:, i]),
    }

print("MÉTRICAS GRU — TEST SET")
print("=" * 55)
for target in TARGET_COLS:
    m = cur_metrics[target]
    print(f"  {target:25s}  MAE={m['MAE']:8.3f}  RMSE={m['RMSE']:8.3f}")

mean_mae  = float(np.mean([m["MAE"]  for m in cur_metrics.values()]))
mean_rmse = float(np.mean([m["RMSE"] for m in cur_metrics.values()]))
print(f"\n  mean_mae  : {mean_mae:.3f}")
print(f"  mean_rmse : {mean_rmse:.3f}")

## 8. Visualizaciones

Cada gráfica se guarda en `SALIDAS modelo GRU/` con sufijo `_v2`.

In [ ]:
## 8.1 Curvas de entrenamiento (loss / MAE / RMSE / R²)

n_ep = len(history.history["loss"])
epochs_ran = range(1, n_ep + 1)
best_ep = int(np.argmin(history.history["val_loss"])) + 1
step = max(1, n_ep // 10)

metrics_cfg = [
    ("loss", "Huber Loss", False),
    ("mae",  "MAE",         False),
    ("rmse", "RMSE",        False),
    ("r2",   "R² Score",    True),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()
for ax, (metric, label, _) in zip(axes, metrics_cfg):
    if metric not in history.history:
        ax.text(0.5, 0.5, f"{label}\nno disponible", ha="center", va="center",
                transform=ax.transAxes, fontsize=11, color="gray")
        ax.set_title(label, fontsize=12)
        continue
    ax.plot(epochs_ran, history.history[metric],         label="train", color="#1565C0", linewidth=2)
    ax.plot(epochs_ran, history.history[f"val_{metric}"], label="val",   color="#E65100", linewidth=2)
    ax.axvline(best_ep, color="green", linestyle="--", linewidth=1.2, alpha=0.7,
               label=f"mejor epoch ({best_ep})")
    ax.set_title(label, fontsize=12)
    ax.set_xlabel("Epoch")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(list(epochs_ran)[::step])

plt.suptitle(f"GRU — Curvas de entrenamiento  |  mejor epoch: {best_ep}/{n_ep}", fontsize=13)
plt.tight_layout()
out_path = f"{OUT_DIR_PLOTS}/fase5_training_curve_gru.png"
plt.savefig(out_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Guardado: {out_path}")

In [ ]:
## 8.2 Predicción vs Real en el tiempo (3 días, un sensor representativo)
# Elijo el sensor con más filas en test para asegurar continuidad temporal.

SENSOR_DEMO = test_df["id"].value_counts().index[0]
mask_sensor = (test_df["id"] == SENSOR_DEMO).values
idx_demo = np.where(mask_sensor)[0][:288]   # 288 × 15 min = 3 días

times_demo  = test_df.loc[mask_sensor, "datetime"].values[:288]
y_true_demo = y_test[idx_demo]
y_pred_demo = y_pred[idx_demo]

fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
for ax, i, target in zip(axes, range(3), TARGET_COLS):
    ax.plot(times_demo, y_true_demo[:, i], label="real",       color="#1565C0", linewidth=1.5)
    ax.plot(times_demo, y_pred_demo[:, i], label="predicción", color="#E65100", linewidth=1.5, alpha=0.85)
    ax.set_title(f"{target} — sensor {SENSOR_DEMO} (3 días de test)", fontsize=11)
    ax.set_ylabel(target)
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel("datetime")

plt.suptitle("GRU — Predicción vs Real", fontsize=13)
plt.tight_layout()
out_path = f"{OUT_DIR_PLOTS}/pred_vs_real_gru.png"
plt.savefig(out_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Guardado: {out_path}")

In [ ]:
## 8.3 Scatter pred vs real (submuestreo a 50k puntos)

rng = np.random.default_rng(42)
N_SCATTER = min(50_000, len(y_test))
idx_s = rng.choice(len(y_test), size=N_SCATTER, replace=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, i, target in zip(axes, range(3), TARGET_COLS):
    yt = y_test[idx_s, i]
    yp = y_pred[idx_s, i]
    ax.scatter(yt, yp, s=2, alpha=0.15, color="#1565C0")
    lim = [min(yt.min(), yp.min()), max(yt.max(), yp.max())]
    ax.plot(lim, lim, color="red", linestyle="--", linewidth=1, label="y = x")
    ax.set_xlabel(f"{target} real")
    ax.set_ylabel(f"{target} predicción")
    ax.set_title(f"{target}  (MAE={cur_metrics[target]['MAE']:.2f})", fontsize=11)
    ax.legend(loc="upper left")
    ax.grid(True, alpha=0.3)

plt.suptitle(f"GRU — Scatter predicción vs real (test, {N_SCATTER:,} puntos)", fontsize=13)
plt.tight_layout()
out_path = f"{OUT_DIR_PLOTS}/scatter_pred_real_gru.png"
plt.savefig(out_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Guardado: {out_path}")

In [ ]:
## 8.4 Histograma de residuos con KDE

from scipy.stats import gaussian_kde

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, i, target in zip(axes, range(3), TARGET_COLS):
    resid = y_pred[:, i] - y_test[:, i]
    counts, bins, _ = ax.hist(resid, bins=80, color="#90A4AE", edgecolor="white", alpha=0.75, density=True)
    # KDE sobre una muestra para velocidad (gaussian_kde es O(n²))
    sample = resid if len(resid) <= 100_000 else rng.choice(resid, 100_000, replace=False)
    kde = gaussian_kde(sample)
    xs = np.linspace(bins[0], bins[-1], 400)
    ax.plot(xs, kde(xs), color="#B71C1C", linewidth=2, label="KDE")
    ax.axvline(0, color="black", linestyle=":", linewidth=1)
    ax.set_title(f"{target}\nμ={resid.mean():+.3f}  σ={resid.std():.3f}", fontsize=11)
    ax.set_xlabel("residuo (pred − real)")
    ax.set_ylabel("densidad")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("GRU — Distribución de residuos (test set)", fontsize=13)
plt.tight_layout()
out_path = f"{OUT_DIR_PLOTS}/residuos_histograma_gru.png"
plt.savefig(out_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Guardado: {out_path}")

In [ ]:
## 8.5 MAE por hora del día

hours_test = test_df["datetime"].dt.hour.values
err_abs = np.abs(y_pred - y_test)

mae_por_hora = np.zeros((24, 3), dtype="float32")
for h in range(24):
    mask = (hours_test == h)
    if mask.sum() > 0:
        mae_por_hora[h] = err_abs[mask].mean(axis=0)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, i, target in zip(axes, range(3), TARGET_COLS):
    ax.bar(range(24), mae_por_hora[:, i], color="#1565C0", edgecolor="white")
    ax.set_title(target, fontsize=11)
    ax.set_xlabel("hora del día")
    ax.set_ylabel("MAE")
    ax.set_xticks(range(0, 24, 2))
    ax.grid(True, alpha=0.3, axis="y")

plt.suptitle("GRU — MAE por hora del día (test set)", fontsize=13)
plt.tight_layout()
out_path = f"{OUT_DIR_PLOTS}/error_por_hora_gru.png"
plt.savefig(out_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Guardado: {out_path}")

In [ ]:
## 8.6 Comparativa de TODOS los modelos (barras MAE + RMSE)
# Carga las métricas de cada modelo disponible y las compara en igualdad de
# condiciones (mismo split de test, mismas features). El modelo actual se toma de
# memoria; el resto, de sus JSON. Así la comparativa funcione aunque falte alguno.

CANDIDATES = [
    ("LSTM v1",     "outputs/results_lstm_original.json"),
    ("BiLSTM v2",   "SALIDAS modelo 2/metricas_v2.json"),
    ("GRU",         "SALIDAS modelo GRU/metricas_gru.json"),
    ("Transformer", "SALIDAS modelo Transformer/metricas_transformer.json"),
]

all_metrics = {}
for label, path in CANDIDATES:
    if label == "GRU":
        continue   # el actual se añade desde memoria (su JSON aún no se ha escrito)
    if os.path.exists(path):
        with open(path) as f:
            d = json.load(f)
        all_metrics[label] = {t: {"MAE": d["targets"][t]["mae"],
                                    "RMSE": d["targets"][t]["rmse"]} for t in TARGET_COLS}

# Modelo actual (en memoria)
all_metrics["GRU"] = {t: {"MAE": cur_metrics[t]["MAE"],
                                "RMSE": cur_metrics[t]["RMSE"]} for t in TARGET_COLS}

labels = list(all_metrics.keys())
print("COMPARATIVA DE MODELOS — TEST SET")
print("=" * 70)
for metric in ["MAE", "RMSE"]:
    print(f"\n  {metric}")
    header = "    " + "Target".ljust(22) + "".join(f"{l:>14}" for l in labels)
    print(header)
    for t in TARGET_COLS:
        row = "    " + t.ljust(22) + "".join(f"{all_metrics[l][t][metric]:>14.3f}" for l in labels)
        print(row)

# --- Gráfico de barras agrupadas: filas = MAE/RMSE, columnas = target ---
import matplotlib.cm as cm
colors = cm.get_cmap("tab10")(np.linspace(0, 1, len(labels)))
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Comparativa de modelos — TEST SET", fontsize=14)
x = np.arange(len(labels))
for col, target in enumerate(TARGET_COLS):
    for row, metric in enumerate(["MAE", "RMSE"]):
        ax = axes[row, col]
        vals = [all_metrics[l][target][metric] for l in labels]
        bars = ax.bar(x, vals, color=colors, edgecolor="white", width=0.6)
        ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20, ha="right", fontsize=8)
        ax.set_title(f"{target} — {metric}", fontsize=11)
        ax.set_ylabel(metric)
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.2f}",
                    ha="center", va="bottom", fontsize=8)
        ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
out_path = f"{OUT_DIR_PLOTS}/comparativa_modelos_gru.png"
plt.savefig(out_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"\nGuardado: {out_path}")

## 9. Guardar métricas JSON (mismo formato que el v1)

In [ ]:
epochs_ran    = len(history.history["loss"])
stopped_early = epochs_ran < EPOCHS

results = {
    "model_name": "gru_stacked",
    "targets": {
        t: {"mae":  round(cur_metrics[t]["MAE"],  6),
             "rmse": round(cur_metrics[t]["RMSE"], 6)}
        for t in TARGET_COLS
    },
    "mean_mae":       round(mean_mae,  6),
    "mean_rmse":      round(mean_rmse, 6),
    "epochs_trained": epochs_ran,
    "stopped_early":  stopped_early,
    "config": {
        "seq_len":       SEQ_LEN,
        "n_features":    N_FEATURES,
        "batch_size":    BATCH_SIZE,
        "epochs_max":    EPOCHS,
        "learning_rate": 5e-5,
        "weight_decay":  1e-4,
        "loss":          "Huber(delta=1.0)",
        "architecture":  "GRU(128) -> Dropout(0.3) -> GRU(64) -> Dropout(0.3) -> Dense(64) -> Dense(32) -> Dense(3)",
        "dropout":       0.4,
        "early_stopping_patience":  15,
        "reduce_lr_patience":       4,
        "reduce_lr_factor":         0.5,
        "reduce_lr_min_lr":         1e-7,
    },
    "checkpoints": {"best": CKPT_BEST, "last": CKPT_LAST, "state": STATE_PATH},
}

json_path = os.path.join(OUT_DIR_PLOTS, "metricas_gru.json")
with open(json_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"Métricas guardadas en {json_path}")
print(json.dumps(results, indent=2))
print("\nFASE Modelo GRU completada.")